# Pipeline Skripsi RFM — K-Means vs Hierarchical

Notebook ini menjalankan seluruh tahapan dari eksplorasi dataset sampai evaluasi dua algoritma. Dataset CSV tidak disimpan di GitHub; notebook akan mencari file di folder Colab atau cache KaggleHub.

In [ ]:
from pathlib import Path
import os, subprocess, sys, shutil

REPO_URL = 'https://github.com/rahmawnj/skripsi-RFM-Algoritma-K-Means-and-hirarical.git'
REPO_DIR = Path('/content/skripsi-RFM-Algoritma-K-Means-and-hirarical')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('Folder kerja:', Path.cwd())
print('File Python:', len(list(REPO_DIR.glob('*.py'))))

In [ ]:
!pip -q install -r requirements.txt
print('Library siap digunakan.')

In [ ]:
DATA_NAME = 'Electronic_sales_Sep2023-Sep2024.csv'
candidates = [
    REPO_DIR / DATA_NAME,
    Path('/content') / DATA_NAME,
]
cache_root = Path.home() / '.cache/kagglehub/datasets/cameronseamons/electronic-sales-sep2023-sep2024/versions'
if cache_root.exists():
    candidates.extend(cache_root.glob(f'*/{DATA_NAME}'))

dataset_path = next((p for p in candidates if p.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError(
        'CSV belum ditemukan. Upload/copy Electronic_sales_Sep2023-Sep2024.csv '
        'ke panel Files Colab atau download dahulu menggunakan KaggleHub.'
    )

if dataset_path.resolve() != (REPO_DIR / DATA_NAME).resolve():
    shutil.copy2(dataset_path, REPO_DIR / DATA_NAME)
    dataset_path = REPO_DIR / DATA_NAME

print('Dataset siap:', dataset_path)
print('Ukuran file (MB):', round(dataset_path.stat().st_size / (1024**2), 2))
print('Catatan: CSV hanya disimpan di runtime Colab dan tidak otomatis di-push ke GitHub.')

## Jalankan seluruh tahapan

Jalankan cell berikut sampai selesai. Jika salah satu tahap error, hentikan dan periksa pesan error sebelum melanjutkan.

In [ ]:
scripts = [
    '01_eksplorasi_dataset.py',
    '02_perhitungan_rfm.py',
    '03_preprocessing.py',
    '04_kmeans.py',
    '05_hierarchical.py',
    '06_evaluasi_perbandingan.py',
]

for script in scripts:
    print('\n' + '=' * 85)
    print('MENJALANKAN:', script)
    print('=' * 85)
    subprocess.run([sys.executable, script], cwd=REPO_DIR, check=True)

print('\nSEMUA TAHAP SELESAI.')

## Ringkasan hasil dan visualisasi

Bagian ini menampilkan kembali tabel hasil dan grafik PNG yang dibuat oleh skrip.

In [ ]:
import pandas as pd
from IPython.display import display, Image

for filename in ['audit_kualitas_data.csv', 'distribusi_order_status.csv', 'hasil_rfm.csv']:
    path = REPO_DIR / filename
    if path.exists():
        print('\n', filename)
        display(pd.read_csv(path).head(10))

for filename in ['kmeans_metrics.csv', 'hierarchical_metrics.csv',
                 'perbandingan_algoritma_semua_k.csv', 'perbandingan_k_terbaik.csv',
                 'profil_cluster_kmeans.csv', 'profil_cluster_hierarchical.csv']:
    path = REPO_DIR / filename
    if path.exists():
        print('\n', filename)
        display(pd.read_csv(path))

images = [
    'visualisasi_distribusi_rfm.png', 'visualisasi_preprocessing.png',
    'grafik_elbow_kmeans.png', 'grafik_silhouette_kmeans.png',
    'dendrogram_hierarchical.png', 'grafik_silhouette_perbandingan.png',
    'grafik_dbi_perbandingan.png', 'grafik_waktu_perbandingan.png',
    'scatter_kmeans.png', 'scatter_hierarchical.png', 'grafik_ukuran_cluster.png',
]
for filename in images:
    path = REPO_DIR / filename
    if path.exists():
        print('\n' + filename)
        display(Image(filename=str(path)))

report = REPO_DIR / 'ringkasan_perbandingan.txt'
if report.exists():
    print('\nRINGKASAN PERBANDINGAN')
    print(report.read_text(encoding='utf-8'))

In [ ]:
# Gabungkan seluruh grafik menjadi SATU gambar PNG dengan judul dan keterangan tiap grafik
from pathlib import Path
from PIL import Image as PILImage, ImageDraw, ImageFont
from IPython.display import display
import ipywidgets as widgets

chart_info = {
    'visualisasi_distribusi_rfm.png': ('1. DISTRIBUSI DATA RFM', 'Sebaran nilai Recency, Frequency, dan Monetary sebelum preprocessing.'),
    'visualisasi_preprocessing.png': ('2. HASIL PREPROCESSING DATA RFM', 'Visualisasi perubahan distribusi setelah transformasi log dan StandardScaler.'),
    'grafik_elbow_kmeans.png': ('3. HASIL ELBOW METHOD — K-MEANS', 'Inertia untuk setiap jumlah cluster (k). Cari titik siku sebagai indikasi kandidat jumlah cluster.'),
    'grafik_silhouette_kmeans.png': ('4. HASIL SILHOUETTE — K-MEANS', 'Mengukur kekompakan cluster dan jarak antar-cluster. Nilai lebih tinggi umumnya lebih baik.'),
    'dendrogram_hierarchical.png': ('5. DENDROGRAM — HIERARCHICAL WARD', 'Struktur penggabungan cluster pada metode Hierarchical dengan Ward linkage.'),
    'grafik_silhouette_perbandingan.png': ('6. PERBANDINGAN SILHOUETTE K-MEANS VS HIERARCHICAL', 'Perbandingan kualitas pemisahan cluster pada jumlah k yang sama.'),
    'grafik_dbi_perbandingan.png': ('7. PERBANDINGAN DAVIES-BOULDIN INDEX (DBI)', 'DBI yang lebih rendah umumnya menunjukkan cluster lebih kompak dan terpisah.'),
    'grafik_waktu_perbandingan.png': ('8. PERBANDINGAN WAKTU KOMPUTASI', 'Waktu komputasi masing-masing algoritma; perhatikan definisi pengukuran pada laporan.'),
    'scatter_kmeans.png': ('9. VISUALISASI CLUSTER K-MEANS', 'Proyeksi PCA 2D untuk membantu melihat pola cluster; PCA hanya untuk visualisasi.'),
    'scatter_hierarchical.png': ('10. VISUALISASI CLUSTER HIERARCHICAL', 'Proyeksi PCA 2D untuk membantu melihat pola cluster; PCA hanya untuk visualisasi.'),
    'grafik_ukuran_cluster.png': ('11. DISTRIBUSI JUMLAH ANGGOTA CLUSTER', 'Jumlah pelanggan pada setiap cluster hasil pengelompokan.'),
}

png_files = [REPO_DIR / name for name in chart_info if (REPO_DIR / name).is_file()]
output_image = REPO_DIR / 'semua_visualisasi_rfm.png'

if not png_files:
    print('Belum ada gambar PNG. Jalankan pipeline sampai selesai terlebih dahulu.')
else:
    canvas_width = 1600
    margin = 55
    title_height = 58
    description_height = 48
    gap = 48
    try:
        title_font = ImageFont.truetype('DejaVuSans-Bold.ttf', 30)
        description_font = ImageFont.truetype('DejaVuSans.ttf', 21)
    except OSError:
        title_font = ImageFont.load_default()
        description_font = ImageFont.load_default()

    prepared = []
    total_height = margin
    for path in png_files:
        with PILImage.open(path) as original:
            img = original.convert('RGB')
            max_width = canvas_width - (2 * margin)
            if img.width > max_width:
                new_height = int(img.height * max_width / img.width)
                img = img.resize((max_width, new_height), PILImage.Resampling.LANCZOS)
            heading, description = chart_info[path.name]
            prepared.append((heading, description, img.copy()))
            total_height += title_height + description_height + img.height + gap

    sheet = PILImage.new('RGB', (canvas_width, total_height + margin), 'white')
    draw = ImageDraw.Draw(sheet)
    y = margin
    for heading, description, img in prepared:
        draw.text((margin, y), heading, fill='#17324D', font=title_font)
        y += title_height
        draw.text((margin, y), description, fill='#444444', font=description_font)
        y += description_height
        x = (canvas_width - img.width) // 2
        sheet.paste(img, (x, y))
        y += img.height + gap

    sheet.save(output_image, format='PNG', optimize=True)
    download_button = widgets.Button(
        description='Download 1 gambar gabungan berlabel',
        button_style='success', icon='download'
    )
    download_status = widgets.Output()

    def download_visualization_image(_):
        from google.colab import files
        with download_status:
            download_status.clear_output()
            files.download(str(output_image))

    download_button.on_click(download_visualization_image)
    display(download_button, download_status)
    print(f'Berhasil menggabungkan {len(png_files)} grafik dengan judul dan keterangan.')
    print(f'Ukuran gambar: {sheet.width} x {sheet.height} piksel')
    display(PILImage.open(output_image))

## Catatan penting untuk skripsi

- Hasil GitHub Actions hanya menguji pipeline menggunakan data sintetis; angka penelitian harus berasal dari CSV asli.
- Periksa asumsi satu baris = satu pesanan dan definisi `Total Price` + `Add-on Total` sebelum menetapkan RFM final.
- Hierarchical Ward standar membutuhkan memori besar; skrip memiliki batas pengaman 12.000 pelanggan.
- Jangan mengubah atau menulis kesimpulan penelitian sebelum tabel, grafik, dan profil cluster dari dataset asli diamati.